# The 5 Things LLMs Actually Do — Applied Prompt Patterns

**Companion notebook for V014 (Phase 3 — Prompt Engineering).**

Prompting isn't one skill. Every job you hand an LLM is really one of **five verbs**:

| # | Verb | Job | The signal ("when do I reach for it?") |
|---|------|-----|----------------------------------------|
| 1 | **Extraction** | text → structured fields | *"I need data OUT of text."* |
| 2 | **Classification** | text → fixed labels | *"I need to sort this into known buckets."* |
| 3 | **Transformation** | text → same meaning, new form | *"Same content, different shape."* |
| 4 | **Generation** | input → new content | *"I need something that didn't exist before."* |
| 5 | **Decomposition** | big request → plan → route to 1–4 | *"This is too big for one prompt."* |

The first four are **atomic** (one prompt, one job). The fifth **chains** them — and chained prompts where the output of one feeds the next are the seed of an *agent*.

We run all five on **one dataset**: a pile of messy customer support tickets (`tickets.json`).

> **Decisions locked for this video:** prompts stay **inline** in the code (prompt-versioning is a future topic), and we use the **free Groq API + Llama** so anyone can follow without paying.


## 0. Setup

We use the **free** [Groq](https://console.groq.com) API, which is OpenAI-compatible and runs open models like Llama very fast.

1. Create a free key at `console.groq.com`.
2. Set it as an environment variable before launching Jupyter:

```bash
export GROQ_API_KEY="gsk_..."
```

3. Install the client (run the next cell once).

> Same idea as the *APIs Explained* video — we send a `messages` array, we get a response back. The only new thing here is **what** we put in the prompt.


In [ ]:
# Run once to install the Groq client
%pip install -q groq

In [ ]:
import json
import os
from pathlib import Path

from groq import Groq

MODEL = "llama-3.3-70b-versatile"
client = Groq()  # reads GROQ_API_KEY from the environment

assert os.environ.get("GROQ_API_KEY"), "Set GROQ_API_KEY first (see the setup cell above)."


def chat(prompt: str, *, json_mode: bool = False, temperature: float = 0.0) -> str:
    """Tiny helper so every pattern below is just: write the prompt, call chat()."""
    kwargs = {}
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        **kwargs,
    )
    return resp.choices[0].message.content


print("Client ready ->", MODEL)

## 1. The data

Ten real-shaped support tickets. Notice the order numbers, dates, and amounts are **buried inside the message text** — that's the whole point. Extraction's job is to dig them out.

In [ ]:
tickets = json.loads(Path("tickets.json").read_text())
print(f"Loaded {len(tickets)} tickets\n")

# Peek at the one from the hook
hook_ticket = tickets[0]
print(hook_ticket["ticket_id"], "-", hook_ticket["customer"])
print(hook_ticket["raw_message"])

## 2. The wrong way — one mega-prompt (the "mush")

First, the mistake almost everyone makes: one vague prompt that tries to do everything at once. Watch it half-extract, half-guess sentiment, and **invent a refund policy we never gave it**.

In [ ]:
mega_prompt = f"""Read this support ticket and handle it.

Ticket:
\"\"\"{hook_ticket['raw_message']}\"\"\""""

print(chat(mega_prompt, temperature=0.7))

That output *sounds* confident, but it's mush: invented policy, no reliable structure, easy to get the order number wrong. The fix is to stop asking for everything at once and name the **verb**.

---

## Pattern 1 — Extraction

**Job:** pull structured facts out of unstructured text.

Two things that make extraction reliable:
- **`temperature=0`** — extraction is not creative; we want the same answer every time.
- **Force a `null` option** — so the model returns `null` instead of *hallucinating* a value when a field genuinely isn't there.

In [ ]:
def extract(ticket_text: str) -> dict:
    prompt = f"""Extract the following fields from the support ticket.
Return ONLY valid JSON, no prose.

Fields:
- order_id (string or null)
- date_mentioned (ISO format YYYY-MM-DD or null)
- amount (number or null)
- product (string or null)

Ticket:
\"\"\"{ticket_text}\"\"\""""
    return json.loads(chat(prompt, json_mode=True))


facts = extract(hook_ticket["raw_message"])
facts

**The honest failure mode.** A ticket with *no* order ID will tempt a weak prompt to invent one. Because we wrote `or null`, it should correctly return `null` here. This is the line that breaks in production if you skip it.

In [ ]:
# TKT-1005 (the SmartScale ramble) has no order id / amount -> expect nulls
no_id_ticket = next(t for t in tickets if t["ticket_id"] == "TKT-1005")
extract(no_id_ticket["raw_message"])

## Pattern 2 — Classification

**Job:** put the text into a box from a *fixed set of labels*.

The trick is **constraining the enum**. If you just ask "what category is this?", the model invents its own labels (`inquiry`, `general`, `misc`) and the output becomes useless for routing. Give it the exact allowed values.

In [ ]:
def classify(ticket_text: str) -> dict:
    prompt = f"""Classify this support ticket.
Return ONLY JSON with these exact fields and allowed values:
- category: one of ["refund","bug","billing","shipping","how_to","cancellation","feature_request"]
- sentiment: one of ["angry","neutral","happy"]
- priority: one of ["low","medium","high"]

Ticket:
\"\"\"{ticket_text}\"\"\""""
    return json.loads(chat(prompt, json_mode=True))


# Route a few tickets and see the priority signal fall out
for t in tickets:
    if t["ticket_id"] in {"TKT-1001", "TKT-1004", "TKT-1010"}:
        print(t["ticket_id"], "->", classify(t["raw_message"]))

`priority: high` falls out for the chargeback threat and the cancellation — that's the signal that decides **which queue** the ticket lands in and **how fast**.

> *Simplifying note:* for a million tickets a day you'd fine-tune a tiny model for this (see the model-selection video). A constrained prompt on a cheap model is plenty to start.

## Pattern 3 — Transformation

**Job:** same meaning, different form. Two flavors back to back.

- **Summarize** the long rambling ticket (TKT-1005) into 2 sentences — but *preserve every distinct issue*, or summaries quietly drop complaints.
- **Translate** the Hindi ticket (TKT-1006) to English so a non-Hindi agent can act on it — keeping order numbers and amounts exact.

In [ ]:
def summarize(ticket_text: str) -> str:
    prompt = f"""Summarize this ticket in at most 2 sentences,
preserving every distinct issue the customer raised.

Ticket:
\"\"\"{ticket_text}\"\"\""""
    return chat(prompt)


ramble = next(t for t in tickets if t["ticket_id"] == "TKT-1005")
print(summarize(ramble["raw_message"]))

In [ ]:
def translate(ticket_text: str) -> str:
    prompt = f"""Translate this ticket to English.
Keep order numbers and amounts exactly as written.

Ticket:
\"\"\"{ticket_text}\"\"\""""
    return chat(prompt)


hindi = next(t for t in tickets if t["ticket_id"] == "TKT-1006")
print("ORIGINAL:\n", hindi["raw_message"], "\n")
print("ENGLISH:\n", translate(hindi["raw_message"]))

## Pattern 4 — Generation

**Job:** create something *new* from the input. Generation has two faces: **language out** and **code out**.

Crucially, we don't let it invent facts like the mega-prompt did — we **ground** it on the extracted facts + a real policy.

In [ ]:
def draft_reply(ticket_text: str, facts: dict, category: str) -> str:
    prompt = f"""You are a support agent for a D2C electronics brand.
Write a reply (max 4 sentences, calm, specific).

Known facts (do not contradict these):
- order_id: {facts.get('order_id')}
- issue category: {category}
- policy: damaged items are eligible for a full refund within 14 days of delivery.

Customer message:
\"\"\"{ticket_text}\"\"\""""
    return chat(prompt, temperature=0.4)


cls = classify(hook_ticket["raw_message"])
print(draft_reply(hook_ticket["raw_message"], facts, cls["category"]))

In [ ]:
def generate_sql(facts: dict) -> str:
    prompt = f"""Generate a single SQL query for this request.
Table: orders(order_id, customer, status, amount, ordered_at, shipped_at)
Return ONLY the SQL, no explanation, no markdown fences.

Request: look up the status and shipped date for order {facts.get('order_id')}."""
    return chat(prompt)


print(generate_sql(facts))

> **⚠️ Guardrail (this breaks in production if you ignore it):** never run model-generated SQL straight against your database. Validate it, use read-only / parameterized access. Generation is the verb you must guardrail the hardest — we'll do that properly in the tools phase.

---

## Pattern 5 — Decomposition (the climax)

**Job:** break a request that's *too big for one prompt* into atomic sub-tasks — each of which is just one of the four verbs above.

Look at **TKT-1009**: *has it shipped? cancel the blue case. refund a priority fee I didn't pick. switch me to monthly billing.* That's four jobs in one message. Throw it at one "handle it" prompt and you're back to the mush.

So step one is itself a prompt: **plan** the work into sub-tasks.

In [ ]:
def decompose(ticket_text: str) -> list[dict]:
    prompt = f"""Break this ticket into a list of atomic sub-tasks.
Return ONLY JSON of the form {{"subtasks": [{{"task": "...", "type": "..."}}]}}
where type is one of ["extraction","classification","transformation","generation"].

Ticket:
\"\"\"{ticket_text}\"\"\""""
    return json.loads(chat(prompt, json_mode=True))["subtasks"]


tangled = next(t for t in tickets if t["ticket_id"] == "TKT-1009")
subtasks = decompose(tangled["raw_message"])
for s in subtasks:
    print(f"[{s['type']:<14}] {s['task']}")

The model didn't *solve* the ticket — it **planned** it. And every sub-task it produced is one of the four atomic verbs we already built. So now we just **route** each sub-task to the right prompt.

In [ ]:
def route(subtask: dict, ticket_text: str, facts: dict) -> dict:
    t = subtask["type"]
    if t == "extraction":
        result = extract(ticket_text)
    elif t == "classification":
        result = classify(ticket_text)
    elif t == "transformation":
        result = summarize(ticket_text)
    elif t == "generation":
        result = chat(
            f"""You are a support agent. Handle ONLY this specific sub-task.
Known facts (do not contradict): {json.dumps(facts)}
Sub-task: {subtask['task']}
Customer message:
\"\"\"{ticket_text}\"\"\"""",
            temperature=0.4,
        )
    else:
        result = f"(unknown type: {t})"
    return {"task": subtask["task"], "type": t, "result": result}


facts_1009 = extract(tangled["raw_message"])
results = [route(s, tangled["raw_message"], facts_1009) for s in subtasks]

for r in results:
    print(f"\n--- {r['type']} :: {r['task']}")
    print(r["result"])

Finally, a last **generation** step stitches the resolved sub-tasks into one clean reply.

In [ ]:
def assemble_reply(ticket_text: str, results: list[dict]) -> str:
    notes = "\n".join(f"- {r['task']}: {r['result']}" for r in results)
    prompt = f"""You are a support agent. Using the resolved sub-task notes below,
write ONE cohesive reply to the customer (max 6 sentences, calm, specific).
Address every point. Do not invent policies or facts.

Resolved sub-tasks:
{notes}

Original message:
\"\"\"{ticket_text}\"\"\""""
    return chat(prompt, temperature=0.4)


print(assemble_reply(tangled["raw_message"], results))

## The pipeline reveal

Stop and look at what we built: a **planner** prompt that splits the work → a **router** that sends each piece to the right specialist prompt → an **assembler** that stitches the final answer. That's the entire video on one screen — and it handles the tangled ticket the mega-prompt couldn't.

Here it is as one function — point it at *any* ticket.

In [ ]:
def handle_ticket(ticket_text: str) -> str:
    """The 5 verbs, composed: decompose -> extract -> route each subtask -> assemble."""
    facts = extract(ticket_text)
    subtasks = decompose(ticket_text)
    results = [route(s, ticket_text, facts) for s in subtasks]
    return assemble_reply(ticket_text, results)


# Run the whole pipeline on the ticket from the hook
print(handle_ticket(hook_ticket["raw_message"]))

## Recap + what's next

```
APPLIED PROMPT PATTERNS — THE 5 VERBS

ATOMIC (one prompt, one job):
  1. EXTRACTION      text -> structured fields      (temp 0, force null)
  2. CLASSIFICATION  text -> fixed labels           (constrain the enum)
  3. TRANSFORMATION  text -> same meaning, new form (summarize / translate)
  4. GENERATION      input -> new content           (ground it; guardrail SQL)

ORCHESTRATOR:
  5. DECOMPOSITION   big request -> plan -> route to 1-4 -> assemble

RULE: don't write one prompt that does everything. Name the verb,
      write the small prompt, chain them when the task is big.
      Chained prompts where output feeds input = an agent.
```

Next time you face any LLM task, ask one question: **which verb is this?** If the answer is "more than one" — decompose.

**The honest boundary (saved for later):** real decomposition gets harder when sub-tasks *depend* on each other (task 2 needs task 1's result). Handling ordering, memory, and retries is exactly the **agent loop** — a whole phase. And notice every step here re-read the *whole* ticket; when the input is a 300-page doc or a 50-turn chat, you run out of room and have to get deliberate about what you feed the model. That's the next video: **Context Engineering**.

> Try it yourself: loop `handle_ticket` over every ticket in `tickets.json`, or swap in your own data. That's how this sticks.
